In [1]:
import os, sys

os.chdir(r"C:\Users\DELL\Desktop\SkillRecommendation-System")
sys.path.insert(0, r"C:\Users\DELL\Desktop\SkillRecommendation-System\projects")

print("Working dir:", os.getcwd())
print("Python path includes projects/:", any("projects" in p for p in sys.path))

Working dir: C:\Users\DELL\Desktop\SkillRecommendation-System
Python path includes projects/: True


In [2]:
import pytest
import pandas as pd
import numpy as np
from data_loader import load_data, build_feature_matrix, get_completed_levels, get_level_pass_rates
from recommendation_engine import RecommendationEngine

# Run tests directly in notebook (no pytest needed)
sample_data = {
    "student_id": ["S1","S1","S1","S2","S2","S2","S3","S3","S3","S4","S4"],
    "level_id":   ["L1","L2","L3","L1","L2","L4","L1","L3","L4","L2","L3"],
    "score":      [80, 75, 90, 70, 65, 85, 60, 55, 80, 90, 88],
    "time_spent_minutes": [30,25,35,40,30,20,50,45,30,25,20],
    "passed":     [1,  1,  1,  1,  1,  1,  1,  0,  1,  1,  1],
}
sample_df = pd.DataFrame(sample_data)
engine = RecommendationEngine(top_k=3, top_n=3, success_threshold=0.5)
engine.fit(sample_df)
full_df = load_data("datasets/student_progress.csv")
full_engine = RecommendationEngine()
full_engine.fit(full_df)

passed = 0
failed = 0

def check(name, condition):
    global passed, failed
    if condition:
        print(f"  PASS  {name}")
        passed += 1
    else:
        print(f"  FAIL  {name}")
        failed += 1

print("=== TestDataLoader ===")
check("load returns DataFrame", isinstance(full_df, pd.DataFrame))
check("required columns present", all(c in full_df.columns for c in ["student_id","level_id","score","time_spent_minutes","passed"]))
check("no nulls in key columns", all(full_df[c].isnull().sum()==0 for c in ["student_id","level_id","score"]))
check("passed column is int", full_df["passed"].dtype in [int, np.int64, np.int32])
fm = build_feature_matrix(sample_df)
check("feature matrix shape (4,4)", fm.shape == (4, 4))
check("feature matrix no negatives", (fm.values >= 0).all())
check("unattempted level is zero", fm.loc["S1","L4"] == 0)
check("completed levels correct", get_completed_levels(sample_df,"S1") == ["L1","L2","L3"])
rates = get_level_pass_rates(sample_df)
check("pass rates between 0 and 1", (rates>=0).all() and (rates<=1).all())

print("\n=== TestRecommendationEngine ===")
check("feature matrix stored after fit", engine.feature_matrix is not None)
check("similarity matrix shape", engine._similarity_matrix.shape == (4,4))
check("similarity diagonal is 1", np.allclose(np.diag(engine._similarity_matrix), 1.0, atol=1e-6))
recs = engine.recommend("S1")
check("recommend returns list", isinstance(recs, list))
check("at most top_n recommendations", len(recs) <= engine.top_n)
check("recommendation keys present", all("level_id" in r and "score" in r and "reason" in r for r in recs))
completed = set(get_completed_levels(sample_df,"S1"))
check("no completed levels recommended", all(r["level_id"] not in completed for r in recs))
check("scores above threshold", all(r["score"] >= engine.success_threshold for r in recs))
check("scores sorted descending", [r["score"] for r in recs] == sorted([r["score"] for r in recs], reverse=True))
try:
    engine.recommend("UNKNOWN")
    check("raises for unknown student", False)
except ValueError:
    check("raises for unknown student", True)
try:
    RecommendationEngine().recommend("S1")
    check("raises before fit", False)
except RuntimeError:
    check("raises before fit", True)

print("\n=== TestIntegration ===")
check("full pipeline runs", isinstance(full_engine.recommend(str(full_df["student_id"].iloc[0])), list))
valid_levels = set(full_df["level_id"].unique())
all_valid = all(rec["level_id"] in valid_levels for sid in full_engine.feature_matrix.index[:10] for rec in full_engine.recommend(sid))
check("recommendations are valid level ids", all_valid)
rng = np.random.default_rng(42)
ids = rng.choice(full_engine.feature_matrix.index.tolist(), size=50, replace=False)
check("70pct threshold on full data", all(rec["score"] >= 0.70 for sid in ids for rec in full_engine.recommend(str(sid))))

print(f"\n{'='*40}")
print(f"Results: {passed} passed, {failed} failed")

=== TestDataLoader ===
  PASS  load returns DataFrame
  PASS  required columns present
  PASS  no nulls in key columns
  PASS  passed column is int
  PASS  feature matrix shape (4,4)
  PASS  feature matrix no negatives
  PASS  unattempted level is zero
  PASS  completed levels correct
  PASS  pass rates between 0 and 1

=== TestRecommendationEngine ===
  PASS  feature matrix stored after fit
  PASS  similarity matrix shape
  PASS  similarity diagonal is 1
  PASS  recommend returns list
  PASS  at most top_n recommendations
  PASS  recommendation keys present
  PASS  no completed levels recommended
  PASS  scores above threshold
  PASS  scores sorted descending
  PASS  raises for unknown student
  PASS  raises before fit

=== TestIntegration ===
  PASS  full pipeline runs
  PASS  recommendations are valid level ids
  PASS  70pct threshold on full data

Results: 23 passed, 0 failed
